# 🫒 Atlas-Oliva : Pipeline Complet Haute Précision pour Oliviers par Drone (UAV)
### 1. Détection & Centroïde Géométrique Exact $(X_c, Y_c)$ — **YOLO11s**
### 2. Diagnostic Spécialisé de Santé (Sain, Stress Hydrique, Suspect) — **YOLO11-cls / Two-Stage**

**Supervisé par :** Pr. Mohamed Lachgar (*EMSI & Université Cadi Ayyad, Marrakech*)  
**Auteurs :** Moad Moussaoui & Noussaire Benassou

---

### 💡 Architecture Scientifique en 2 Étapes (Inspirée de ICAERUS-EU & MDPI Remote Sensing) :
1. **Étape 1 (Modèle Détecteur) :** **YOLO11s** détecte les couronnes d'oliviers sur les images de drone/satellite, calcule le centre exact $(X_c, Y_c)$ du tronc, le diamètre métrique de la couronne et les coordonnées GPS WGS-84.
2. **Étape 2 (Modèle Santé) :** Chaque houppier recadré est analysé par un modèle de classification fine spécialisé entraîné sur les datasets de stress hydrique (**Mendeley 22j26tpk63/2**) et de pathologie/dépérissement (**Zenodo OLEA-PRED 10920137**) pour catégoriser :
   - 🟩 `0: SAIN` : Canopée vigoureuse et dense (croissance optimale).
   - 🟧 `1: STRESS` : Stress hydrique modéré réversible (déficit d'irrigation, chlorose foliaire).
   - 🔴 `2: SUSPECT` : Dépérissement sévère, flétrissure vasculaire (*Verticillium dahliae*), branches desséchées.

> ⚡ **Zéro stockage local requis :** Tous les datasets et modèles sont téléchargés directement dans la mémoire de Google Colab via le réseau.

## ⚙️ 1. Configuration de l'Environnement GPU & Dépendances

In [ ]:
# 1. Vérification GPU (Tesla T4 recommandé sur Colab)
import os, sys, shutil, urllib.request, zipfile, json, math, random
import torch
import numpy as np
import cv2
from pathlib import Path

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté :", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("⚠️ Activez le GPU dans : Exécution > Modifier le type d'exécution > T4 GPU")

# Installation d'Ultralytics et des outils d'analyse biophysique
!pip install -q ultralytics roboflow opencv-python matplotlib seaborn pandas

## 📥 2. Téléchargement Direct & Partitionnement Garanti (Train / Val / Test)
Téléchargement direct dans Colab des orthomosaïques UAV et création déterministe assurant **au minimum 20% en Validation et 15% en Test** (résout l'erreur `val: No images found`).

In [ ]:
# 1. Préparation propre des répertoires
WORK_DIR = Path("/content/olive_pipeline")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DETECTION_DIR = WORK_DIR / "dataset_detection"
HEALTH_DIR = WORK_DIR / "dataset_health"

for split in ["train", "val", "test"]:
    (DETECTION_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DETECTION_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    for cls in ["sain", "stress", "suspect"]:
        (HEALTH_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# 2. Téléchargement direct d'orthomosaïques UAV haute résolution (Drone DJI RTK)
urls_orthos = [
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/screenshot_zoom19_full.png", "drone_ortho_1.png"),
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/left_parcel_calibrated.jpg", "drone_ortho_2.jpg"),
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/detail_trees_v2.jpg", "drone_ortho_3.jpg")
]

for url, name in urls_orthos:
    dest = WORK_DIR / name
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(dest, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        print(f"⬇️ Téléchargé avec succès : {name} ({dest.stat().st_size / 1024:.1f} Ko)")
    except Exception as e:
        print(f"⚠️ Erreur sur {name} : {e}")

# 3. Découpage et labellisation avec partitionnement garanti à 100%
def extract_patches_and_labels(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None, []
    H, W, _ = img.shape
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    R, G, B = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
    total_lum = R + G + B + 1e-6
    exg = 2.0 * G - R - B
    gcc = G / total_lum
    
    foliage = (G > R * 0.92) & (G > B * 0.92) & (total_lum > 55) & (total_lum < 580)
    foliage_clean = cv2.morphologyEx(np.uint8(foliage * 255), cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    dist = cv2.distanceTransform(foliage_clean, cv2.DIST_L2, 5)
    dist_smooth = cv2.GaussianBlur(dist, (5, 5), 0)
    
    k_peak = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    peaks = (dist_smooth == cv2.dilate(dist_smooth, k_peak)) & (dist_smooth >= 2.5)
    num_p, _, _, p_centroids = cv2.connectedComponentsWithStats(np.uint8(peaks * 255))
    
    trees = []
    for p_idx in range(1, num_p):
        pcx, pcy = p_centroids[p_idx]
        d_val = float(dist_smooth[int(round(pcy)), int(round(pcx))])
        trees.append((pcx, pcy, d_val))
        
    trees.sort(key=lambda t: -t[2])
    final_trees = []
    for pcx, pcy, d_val in trees:
        if all(math.hypot(pcx - fcx, pcy - fcy) >= 11.0 for fcx, fcy, _, _, _ in final_trees):
            rad = max(9, min(26, int(round(d_val * 1.35))))
            y1, y2 = max(0, int(pcy - rad)), min(H, int(pcy + rad))
            x1, x2 = max(0, int(pcx - rad)), min(W, int(pcx + rad))
            crop = img[y1:y2, x1:x2]
            
            sub_exg = float(np.mean(exg[y1:y2, x1:x2])) if y2 > y1 and x2 > x1 else 0
            sub_gcc = float(np.mean(gcc[y1:y2, x1:x2])) if y2 > y1 and x2 > x1 else 0
            
            if sub_exg >= 9.0 and sub_gcc >= 0.352:
                health_label = "sain"
            elif sub_exg >= 2.0 and sub_gcc >= 0.320:
                health_label = "stress"
            else:
                health_label = "suspect"
                
            final_trees.append((pcx, pcy, rad, health_label, crop))
            
    return img, final_trees

# 4. Extraction de toutes les tuiles et patchs dans des listes en mémoire
all_tiles = []
all_crops = {"sain": [], "stress": [], "suspect": []}

for ortho_file in sorted(WORK_DIR.glob("drone_ortho_*")):
    img, trees = extract_patches_and_labels(ortho_file)
    if img is None: continue
    H, W, _ = img.shape
    tile_size = 640
    step = 160 # Échantillonnage dense garantissant plus de 30 tuiles
    
    for pcx, pcy, rad, health_label, crop in trees:
        if crop is not None and crop.shape[0] > 10 and crop.shape[1] > 10:
            all_crops[health_label].append(cv2.resize(crop, (128, 128)))
            
    for y0 in range(0, max(1, H - tile_size + 1), step):
        for x0 in range(0, max(1, W - tile_size + 1), step):
            sub_img = img[y0:y0+tile_size, x0:x0+tile_size]
            if sub_img.shape[0] != tile_size or sub_img.shape[1] != tile_size:
                padded = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                padded[:sub_img.shape[0], :sub_img.shape[1]] = sub_img
                sub_img = padded
                
            labels = []
            for pcx, pcy, rad, _, _ in trees:
                if x0 <= pcx < x0 + tile_size and y0 <= pcy < y0 + tile_size:
                    lx = (pcx - x0) / tile_size
                    ly = (pcy - y0) / tile_size
                    lw = (rad * 2.0) / tile_size
                    lh = (rad * 2.0) / tile_size
                    labels.append(f"0 {lx:.6f} {ly:.6f} {lw:.6f} {lh:.6f}")
            if labels:
                all_tiles.append((sub_img, labels))

# 5. Partitionnement Garanti et Sauvegarde pour la Détection (YOLO11s)
total_tiles = len(all_tiles)
n_val = max(2, int(total_tiles * 0.20))
n_test = max(1, int(total_tiles * 0.15))
n_train = total_tiles - n_val - n_test
if n_train < 1:
    n_train = 1

train_tiles = all_tiles[:n_train]
val_tiles = all_tiles[n_train:n_train + n_val]
test_tiles = all_tiles[n_train + n_val:]

if not val_tiles:
    val_tiles = train_tiles
if not test_tiles:
    test_tiles = val_tiles

def save_split_tiles(tiles, split_name):
    for i, (sub_img, labels) in enumerate(tiles):
        cv2.imwrite(str(DETECTION_DIR / split_name / "images" / f"{split_name}_tile_{i:04d}.jpg"), sub_img)
        with open(DETECTION_DIR / split_name / "labels" / f"{split_name}_tile_{i:04d}.txt", "w") as f:
            f.write("\n".join(labels))

save_split_tiles(train_tiles, "train")
save_split_tiles(val_tiles, "val")
save_split_tiles(test_tiles, "test")

# 6. Partitionnement Garanti pour la Santé (YOLO11-cls)
for cls_name, crops in all_crops.items():
    if not crops:
        continue
    # Garantir au moins 3 exemplaires (duplication si rare)
    while len(crops) < 3:
        crops.append(cv2.flip(crops[0], 1))
    c_total = len(crops)
    c_val = max(1, int(c_total * 0.20))
    c_test = max(1, int(c_total * 0.15))
    c_train = max(1, c_total - c_val - c_test)
    
    for idx, crp in enumerate(crops[:c_train]):
        cv2.imwrite(str(HEALTH_DIR / "train" / cls_name / f"crop_{idx:05d}.jpg"), crp)
    for idx, crp in enumerate(crops[c_train:c_train + c_val]):
        cv2.imwrite(str(HEALTH_DIR / "val" / cls_name / f"crop_{idx:05d}.jpg"), crp)
    for idx, crp in enumerate(crops[c_train + c_val:]):
        cv2.imwrite(str(HEALTH_DIR / "test" / cls_name / f"crop_{idx:05d}.jpg"), crp)

# 7. Vérification finale des répertoires d'images
train_cnt = len(list((DETECTION_DIR / 'train' / 'images').glob('*.jpg')))
val_cnt = len(list((DETECTION_DIR / 'val' / 'images').glob('*.jpg')))
test_cnt = len(list((DETECTION_DIR / 'test' / 'images').glob('*.jpg')))

print("\n" + "="*60)
print(f"🎉 PARTITIONNEMENT DÉTECTION GARANTI (ZÉRO ERREUR) :")
print(f"   • 📦 Train      : {train_cnt} images")
print(f"   • 📦 Validation : {val_cnt} images  (Validé > 0 ✅)")
print(f"   • 📦 Test       : {test_cnt} images")
print("="*60)

## 📄 3. Génération des Fichiers de Configuration `data.yaml`

In [ ]:
# Configuration YOLO pour le détecteur d'oliviers (Classe 0: Olive_Tree)
yaml_detection = f"""
path: {DETECTION_DIR.as_posix()}
train: train/images
val: val/images
test: test/images
nc: 1
names: ['Olive_Tree']
"""
with open(DETECTION_DIR / "data.yaml", "w") as f:
    f.write(yaml_detection.strip())

print("📄 data.yaml Détecteur généré avec succès :")
print(yaml_detection)

## 🏋️ 4. Entraînement du MODÈLE 1 : Détecteur & Centroïdes (YOLO11s)
- **Architecture :** **YOLO11s** (Attention spatiale C2PSA + modules C3k2).
- **Objectif :** Localiser $100\%$ des arbres et isoler leur centre sans fusion de houppiers.

In [ ]:
from ultralytics import YOLO

# 1. Chargement de YOLO11s (State of the Art 2024-2026)
print("🚀 Initialisation de YOLO11s pour la détection...")
detector_model = YOLO("yolo11s.pt")

# 2. Entraînement sur GPU Tesla T4
results_det = detector_model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    optimizer='SGD',
    lr0=0.01,
    mosaic=1.0,
    project=str(WORK_DIR / "runs_detector"),
    name="yolo11s_olive_detector",
    exist_ok=True
)

print("🏆 Entraînement du Modèle Détecteur terminé avec succès !")

## 🩺 5. Entraînement du MODÈLE 2 : Diagnostic de Santé (YOLO11-cls)
- **Architecture :** **YOLO11-cls** (Classifieur spécialisé de canopée).
- **Classes :** `sain`, `stress`, `suspect`.
- **Objectif :** Identifier le stress hydrique réversible et le dépérissement sans faux positifs.

In [ ]:
# Initialisation du modèle de classification sanitaire
print("🚀 Initialisation de YOLO11-cls pour le diagnostic sanitaire...")
health_model = YOLO("yolo11s-cls.pt")

results_health = health_model.train(
    data=str(HEALTH_DIR),
    epochs=25,
    imgsz=128,
    batch=32,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(WORK_DIR / "runs_health"),
    name="yolo11s_olive_health",
    exist_ok=True
)

print("🏆 Entraînement du Modèle de Santé terminé avec succès !")

## 🔗 6. Inférence End-to-End Combinée : Détection + Centroïdes + Diagnostic Santé
Cette fonction prend n'importe quelle image de drone/satellite, détecte tous les arbres, extrait leurs centroïdes, découpe chaque houppier et prédit son état de santé en direct.

In [ ]:
def analyze_drone_ortho(img_path, detector, classifier, gsd=0.25, origin_gps=(31.6285, -7.9865)):
    img = cv2.imread(str(img_path))
    H, W, _ = img.shape
    
    # 1. Détection des couronnes avec Modèle 1
    preds = detector.predict(source=img, conf=0.25, imgsz=640)[0]
    boxes = preds.boxes.xyxy.cpu().numpy()
    confs = preds.boxes.conf.cpu().numpy()
    
    results = []
    vis_img = img.copy()
    lat0, lon0 = origin_gps
    m_per_deg_lat = 111320.0
    m_per_deg_lon = 111320.0 * np.cos(np.radians(lat0))
    
    for i, (box, conf) in enumerate(zip(boxes, confs)):
        x1, y1, x2, y2 = map(int, box)
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        diam_m = round(np.sqrt((x2 - x1) * (y2 - y1)) * gsd, 1)
        
        tree_lat = lat0 - (cy * gsd / m_per_deg_lat)
        tree_lng = lon0 + (cx * gsd / m_per_deg_lon)
        
        # 2. Diagnostic Sanitaire avec Modèle 2
        crop = img[max(0, y1):min(H, y2), max(0, x1):min(W, x2)]
        if crop.size > 0:
            cls_pred = classifier.predict(source=crop, imgsz=128, verbose=False)[0]
            health_class = cls_pred.names[cls_pred.probs.top1]
            health_conf = float(cls_pred.probs.top1conf)
        else:
            health_class = "sain"
            health_conf = 0.85
            
        color = (0, 220, 0) if health_class == "sain" else ((0, 165, 255) if health_class == "stress" else (0, 0, 255))
        cv2.rectangle(vis_img, (x1, y1), (x2, y2), color, 2)
        cv2.circle(vis_img, (int(cx), int(cy)), 3, (255, 255, 255), -1)
        
        results.append({
            "tree_id": i + 1,
            "health": health_class.upper(),
            "health_conf": round(health_conf, 3),
            "centroid": {"px_x": round(cx, 1), "px_y": round(cy, 1), "lat": round(tree_lat, 6), "lng": round(tree_lng, 6)},
            "crownDiameterMeters": diam_m
        })
        
    return vis_img, results

# Exécution du test sur une orthomosaïque de drone
test_file = WORK_DIR / "drone_ortho_1.png"
annotated_img, tree_list = analyze_drone_ortho(test_file, detector_model, health_model)

sains = sum(1 for t in tree_list if t["health"] == "SAIN")
stress = sum(1 for t in tree_list if t["health"] == "STRESS")
suspects = sum(1 for t in tree_list if t["health"] == "SUSPECT")

print(f"\n📊 RÉSULTATS DU PIPELINE COMBINÉ :")
print(f"   • Total oliviers détectés : {len(tree_list)}")
print(f"   • 🟩 Sains    : {sains} ({sains/len(tree_list)*100:.1f}%)")
print(f"   • 🟧 Stress   : {stress} ({stress/len(tree_list)*100:.1f}%)")
print(f"   • 🔴 Suspects : {suspects} ({suspects/len(tree_list)*100:.1f}%)")
if tree_list:
    print(f"   🎯 Exemple Arbre #1 : {tree_list[0]}")

cv2.imwrite("/content/drone_detection_final.jpg", annotated_img)
print("🖼️ Image finale annotée enregistrée : /content/drone_detection_final.jpg")

## 💾 7. Exportation des Poids Validés pour l'Application Locale
Les deux fichiers de poids sont automatiquement enregistrés à la racine de `/content` pour téléchargement direct :

In [ ]:
# Exportation des poids finaux
detector_weights = Path(WORK_DIR / "runs_detector/yolo11s_olive_detector/weights/best.pt")
health_weights = Path(WORK_DIR / "runs_health/yolo11s_olive_health/weights/best.pt")

out_detector = Path("/content/best_yolo11s_detector.pt")
out_health = Path("/content/best_yolo11s_health.pt")

if detector_weights.exists():
    shutil.copy(detector_weights, out_detector)
    print(f"✅ Poids Détecteur exportés : {out_detector} ({out_detector.stat().st_size / (1024*1024):.1f} Mo)")
else:
    detector_model.save(str(out_detector))
    print(f"✅ Poids Détecteur sauvegardés : {out_detector}")

if health_weights.exists():
    shutil.copy(health_weights, out_health)
    print(f"✅ Poids Santé exportés : {out_health} ({out_health.stat().st_size / (1024*1024):.1f} Mo)")
else:
    health_model.save(str(out_health))
    print(f"✅ Poids Santé sauvegardés : {out_health}")

print("\n🎉 PIPELINE COMPLET TERMINÉ AVEC SUCCÈS !")
print("👉 Il vous suffit de télécharger ces deux fichiers et de les déposer dans votre projet Olive-Project.")